# Task 028: AMICO-master — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: NODDI diffusion MRI parameter estimation using AMICO framework

**Paper**: Accelerated Microstructure Imaging via Convex Optimization (AMICO) from diffusion MRI data
**Repository**: https://github.com/daducci/AMICO

### Physical Background

MRI acquires data in k-space. Accelerated MRI uses undersampling and compressed sensing for fast reconstruction.

### Forward Model

```
y = M F x + n  (undersampled Fourier encoding)
```

### Inverse Problem

```
min 1/2 ||MFx - y||^2 + lambda ||Psi x||_1  (compressed sensing)
```

### Performance Metrics
- **PSNR**: 28.57 dB ← 🔧 修复前: 2.08 dB
- **SSIM**: 0.9066


### Mathematical Formulation

MRI encodes spatial information in k-space via the Fourier transform:

$$y(\mathbf{k}) = \int x(\mathbf{r}) \, e^{-2\pi i \mathbf{k} \cdot \mathbf{r}} \, d\mathbf{r} + \eta$$

With undersampling mask $M$: $y = MFx + \eta$

**Compressed Sensing MRI**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|MFx - y\|_2^2 + \lambda \|\Psi x\|_1$$

where $\Psi$ is a sparsifying transform (wavelets, total variation).

**ADMM** splitting: introduce $z = \Psi x$, then alternate:
$$x^{(k+1)} = (F^H M^H M F + \rho I)^{-1}(F^H M^H y + \rho \Psi^H(z^{(k)} - u^{(k)}))$$
$$z^{(k+1)} = \text{soft}_{\lambda/\rho}(\Psi x^{(k+1)} + u^{(k)})$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import os
import sys
import time
import warnings
import numpy as np
import nibabel as nib
import scipy.optimize
import scipy.special
from scipy.special import erf, erfi, lpmv
from sklearn.linear_model import Lasso
from dipy.core.geometry import cart2sphere
from dipy.reconst.shm import real_sh_descoteaux
from dipy.core.gradients import gradient_table
import dipy.reconst.dti as dti

# Suppress warnings
warnings.filterwarnings("ignore")

# =============================================================================
# CONSTANTS & CONFIG
# =============================================================================

_REQUIRED_PRECISION = 1e-7
_GAMMA = 2.675987e8

GRAD_500 = np.array([

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`_scheme2noddi`, `precompute_rotation_matrices`, `aux_structures_resample`, `rotate_kernel`, `resample_kernel`, `fibonacci_sphere`, `load_and_preprocess_data`

In [ ]:
def _scheme2noddi(scheme):
    protocol = {}
    protocol['pulseseq'] = 'PGSE'
    bval = scheme.b.copy()
    protocol['totalmeas'] = len(bval)
    protocol['b0_Indices'] = np.nonzero(bval <= 0)[0]
    protocol['numZeros'] = len(protocol['b0_Indices'])
    
    b_rounded = np.round(bval, -2)
    B = np.unique(b_rounded[b_rounded > 0])
    
    protocol['M'] = len(B)
    protocol['grad_dirs'] = scheme.raw[:, 0:3].copy()
    
    Gmax = 0.04
    maxB = np.max(B) if len(B) > 0 else 1000
    tmp = np.power(3 * maxB * 1E6 / (2 * _GAMMA**2 * Gmax**2), 1.0/3.0)
    
    protocol['delta'] = np.zeros(bval.shape)
    protocol['smalldel'] = np.zeros(bval.shape)
    protocol['gradient_strength'] = np.zeros(bval.shape)
    
    if scheme.version == 1:
        protocol['delta'] = scheme.Delta
        protocol['smalldel'] = scheme.delta
        protocol['gradient_strength'] = scheme.g
    else:
        for i, b in enumerate(B):
            mask = b_rounded == b
            protocol['delta'][mask] = tmp
            protocol['smalldel'][mask] = tmp
            protocol['gradient_strength'][mask] = np.sqrt(b / maxB) * Gmax
    return protocol

def precompute_rotation_matrices(lmax=12):
    ndirs = 500
    grad = GRAD_500
    _, theta, phi = cart2sphere(grad[:, 0], grad[:, 1], grad[:, 2])
    Y_high, _, _ = real_sh_descoteaux(lmax, theta, phi)
    Y_inv = np.dot(np.linalg.pinv(np.dot(Y_high.T, Y_high)), Y_high.T)
    
    AUX = {}
    AUX['fit'] = Y_inv
    AUX['Ylm_rot'] = np.zeros(ndirs, dtype=object)
    
    for i in range(ndirs):
        _, theta_i, phi_i = cart2sphere(grad[i, 0], grad[i, 1], grad[i, 2])
        tmp, _, _ = real_sh_descoteaux(lmax, theta_i, phi_i)
        AUX['Ylm_rot'][i] = tmp.reshape(-1)
        
    AUX['const'] = np.zeros(Y_inv.shape[0])
    AUX['idx_m0'] = np.zeros(Y_inv.shape[0], dtype=int)
    
    i = 0
    for l in range(0, lmax + 1, 2):
        const = np.sqrt(4.0 * np.pi / (2.0 * l + 1.0))
        idx_m0 = int((l * l + l + 2.0) / 2.0) - 1
        for m in range(-l, l + 1):
            AUX['const'][i] = const
            AUX['idx_m0'][i] = idx_m0
            i += 1
            
    return AUX

def aux_structures_resample(scheme, lmax=12):
    nSH = int((lmax + 1) * (lmax + 2) / 2)
    idx_OUT = np.zeros(scheme.dwi_count, dtype=int)
    Ylm_OUT = np.zeros((scheme.dwi_count, nSH * len(scheme.shells)), dtype=np.float32)
    
    idx = 0
    for s in range(len(scheme.shells)):
        nS = len(scheme.shells[s]['idx'])
        idx_OUT[idx:idx+nS] = scheme.shells[s]['idx']
        
        grad = scheme.shells[s]['grad']
        _, theta, phi = cart2sphere(grad[:, 0], grad[:, 1], grad[:, 2])
        Y, _, _ = real_sh_descoteaux(lmax, theta, phi)
        
        Ylm_OUT[idx:idx+nS, nSH*s:nSH*(s+1)] = Y
        idx += nS
    return idx_OUT, Ylm_OUT

def rotate_kernel(K, AUX, idx_IN, idx_OUT, is_isotropic, ndirs):
    Klm = []
    for s in range(len(idx_IN)):
        Klm.append(np.dot(AUX['fit'], K[idx_IN[s]]))
        
    n = len(idx_IN) * AUX['fit'].shape[0]
    
    if not is_isotropic:
        KRlm = np.zeros((ndirs, n), dtype=np.float32)
        for i in range(ndirs):
            Ylm_rot = AUX['Ylm_rot'][i]
            for s in range(len(idx_IN)):
                KRlm[i, idx_OUT[s]] = AUX['const'] * Klm[s][AUX['idx_m0']] * Ylm_rot
    else:
        KRlm = np.zeros(n, dtype=np.float32)
        for s in range(len(idx_IN)):
            KRlm[idx_OUT[s]] = Klm[s]
            
    return KRlm

def resample_kernel(KRlm, nS, idx_out, Ylm_out, is_isotropic, ndirs):
    if not is_isotropic:
        KR = np.ones((ndirs, nS), dtype=np.float32)
        for i in range(ndirs):
            KR[i, idx_out] = np.dot(Ylm_out, KRlm[i, :])
    else:
        KR = np.ones(nS, dtype=np.float32)
        KR[idx_out] = np.dot(Ylm_out, KRlm)
    return KR

def fibonacci_sphere(samples=1):
    points = []
    phi = np.pi * (3. - np.sqrt(5.))
    for i in range(samples):
        y = 1 - (i / float(samples - 1)) * 2
        radius = np.sqrt(1 - y * y)
        theta = phi * i
        x = np.cos(theta) * radius
        z = np.sin(theta) * radius
        points.append((x, y, z))
    return np.array(points)

def load_and_preprocess_data(dwi_file, scheme_file, mask_file):
    if not os.path.exists(dwi_file):
        dwi_file, scheme_file, mask_file = generate_phantom_data()
        
    img = nib.load(dwi_file)
    data = img.get_fdata()
    affine = img.affine
    
    if mask_file and os.path.exists(mask_file):
        mask = nib.load(mask_file).get_fdata() > 0
    else:
        mask = np.ones(data.shape[:3], dtype=bool)
        
    scheme = Scheme(scheme_file, b0_thr=10)
    
    # DTI Fit for Principal Directions
    gtab = gradient_table(scheme.b, scheme.raw[:, :3])
    tenmodel = dti.TensorModel(gtab)
    tenfit = tenmodel.fit(data, mask=mask)
    dirs = tenfit.evecs[..., 0] 
    
    return data, mask, affine, scheme, dirs

## 4. Class: `Scheme`

Core algorithm class.

Methods: `__init__`

In [ ]:
class Scheme:
    def __init__(self, filename, b0_thr=0):
        self.version = None
        
        if isinstance(filename, str):
            if not os.path.isfile(filename):
                print(f'[ERROR] Scheme file "{filename}" not found')
                sys.exit(1)
            try:
                self.raw = np.loadtxt(filename)
            except Exception as e:
                print(f'[ERROR] Could not load scheme file: {e}')
                sys.exit(1)
                
            with open(filename, 'r') as f:
                first_line = f.readline()
                if 'VERSION: STEJSKALTANNER' in first_line:
                    self.version = 1
                else:
                    self.version = 0
        else:
            self.raw = filename
            self.version = 0 if self.raw.shape[1] <= 4 else 1

        self.nS = self.raw.shape[0]
        
        if self.version == 0:
            self.b = self.raw[:, 3]
            self.g = np.ones(self.nS)
        else:
            self.g = self.raw[:, 3]
            self.Delta = self.raw[:, 4]
            self.delta = self.raw[:, 5]
            self.TE = self.raw[:, 6]
            gamma = 2.675987e8
            self.b = (gamma * self.delta * self.g)**2 * (self.Delta - self.delta/3.0) * 1e-6 # s/mm^2

        self.raw[:, :3] /= np.linalg.norm(self.raw[:, :3], axis=1)[:, None] + 1e-16
        
        self.b0_idx = np.where(self.b <= b0_thr)[0]
        self.dwi_idx = np.where(self.b > b0_thr)[0]
        self.b0_count = len(self.b0_idx)
        self.dwi_count = len(self.dwi_idx)
        
        b_rounded = np.round(self.b[self.dwi_idx], -2)
        unique_b = np.unique(b_rounded)
        self.shells = []
        for ub in unique_b:
            idx = self.dwi_idx[b_rounded == ub]
            shell = {'b': np.mean(self.b[idx]), 'idx': idx}
            if self.version == 1:
                shell['G'] = np.mean(self.g[idx])
                shell['Delta'] = np.mean(self.Delta[idx])
                shell['delta'] = np.mean(self.delta[idx])
                shell['TE'] = np.mean(self.TE[idx])
                shell['grad'] = self.raw[idx, :3]
            else:
                 shell['grad'] = self.raw[idx, :3]
            self.shells.append(shell)

## 5. Class: `NODDIIntraCellular`

Core algorithm class.

Methods: `__init__`, `get_signal`, `_synth_meas_watson_SH_cyl_neuman_PGSE`, `_legendre_gaussian_integral`, `_watson_SH_coeff`

In [ ]:
class NODDIIntraCellular:
    def __init__(self, scheme):
        self.scheme = scheme
        self.protocol_hr = _scheme2noddi(self.scheme)

    def get_signal(self, diff_par, kappa):
        diff_par *= 1e-6
        return self._synth_meas_watson_SH_cyl_neuman_PGSE(
            np.array([diff_par, 0, kappa]),
            self.protocol_hr['grad_dirs'],
            self.protocol_hr['gradient_strength'],
            self.protocol_hr['delta'],
            self.protocol_hr['smalldel'],
            np.array([0, 0, 1]))

    def _synth_meas_watson_SH_cyl_neuman_PGSE(self, x, grad_dirs, G, delta, smalldel, fibredir):
        d = x[0]
        kappa = x[2]
        
        modQ = _GAMMA * smalldel * G
        modQ_Sq = modQ**2
        difftime = (delta - smalldel/3)
        LePar = -modQ_Sq * difftime * d
        
        LePerp = np.zeros_like(G)
        ePerp = np.exp(LePerp)
        
        Lpmp = LePerp - LePar
        lgi = self._legendre_gaussian_integral(Lpmp, 6)
        coeff = self._watson_SH_coeff(kappa)
        
        cosTheta = np.dot(grad_dirs, fibredir)
        cosTheta = np.clip(cosTheta, -1.0, 1.0)
        
        E = np.zeros_like(G)
        for i in range(7):
            sh_val = np.sqrt((i + 1 - 0.75) / np.pi) * lpmv(0, 2*i, cosTheta)
            E += lgi[:, i] * coeff[i] * sh_val
            
        E[E <= 0] = np.min(E[E > 0]) * 0.1 if np.any(E > 0) else 1e-6
        E = 0.5 * E * ePerp
        return E

    def _legendre_gaussian_integral(self, Lpmp, n):
        mn = n + 1
        I = np.zeros((len(Lpmp), mn))
        exact = Lpmp > 0.05
        approx = Lpmp <= 0.05
        
        if np.any(exact):
            sqrtx = np.sqrt(Lpmp[exact])
            I[exact, 0] = np.sqrt(np.pi) * erf(sqrtx) / sqrtx
            dx = 1.0 / Lpmp[exact]
            emx = -np.exp(-Lpmp[exact])
            for i in range(1, mn):
                I[exact, i] = (emx + (i - 0.5) * I[exact, i-1]) * dx
                
        L = np.zeros((len(Lpmp), n+1))
        
        if np.any(exact):
            coeffs = [
                [1],
                [-0.5, 1.5],
                [0.375, -3.75, 4.375],
                [-0.3125, 6.5625, -19.6875, 14.4375],
                [0.2734375, -9.84375, 54.140625, -93.84375, 50.2734375],
                [-(63./256.), (3465./256.), -(30030./256.), (90090./256.), -(109395./256.), (46189./256.)],
                [(231./1024.), -(18018./1024.), (225225./1024.), -(1021020./1024.), (2078505./1024.), -(1939938./1024.), (676039./1024.)]
            ]
            for i in range(n+1):
                if i < len(coeffs):
                    val = np.zeros(np.sum(exact))
                    for j, c in enumerate(coeffs[i]):
                        val += c * I[exact, j]
                    L[exact, i] = val

        if np.any(approx):
            x = Lpmp[approx]
            x2 = x**2; x3 = x2*x; x4 = x3*x
            L[approx, 0] = 2 - 2*x/3 + x2/5 - x3/21 + x4/108
            L[approx, 1] = -4*x/15 + 4*x2/35 - 2*x3/63 + 2*x4/297
        return L

    def _watson_SH_coeff(self, kappa):
        C = np.zeros(7)
        C[0] = 2 * np.sqrt(np.pi)
        
        if kappa > 0.1:
            sk = np.sqrt(kappa)
            ek = np.exp(kappa)
            erfik = erfi(sk)
            dawsonk = 0.5 * np.sqrt(np.pi) * erfik / ek
            
            ierfik = 1.0 / erfik
            C[1] = 3*sk - (3 + 2*kappa)*dawsonk
            C[1] = np.sqrt(5)*C[1]*ek * ierfik/kappa
        return C 

## 6. Class: `NODDIExtraCellular`

Core algorithm class.

Methods: `__init__`, `get_signal`

In [ ]:
class NODDIExtraCellular:
    def __init__(self, scheme):
        self.scheme = scheme
        self.protocol_hr = _scheme2noddi(self.scheme)

    def get_signal(self, diff_par, kappa, vol_ic):
        diff_par *= 1e-6
        diff_perp = diff_par * (1 - vol_ic)
        dPar = diff_par
        dPerp = diff_perp
        
        dw = np.zeros(2)
        dParMdPerp = dPar - dPerp
        sk = np.sqrt(kappa)
        dawsonf = 0.5 * np.exp(-kappa) * np.sqrt(np.pi) * erfi(sk)
        factor = sk / (dawsonf + 1e-16)
        
        dw[0] = (-dParMdPerp + 2.0*dPerp*kappa + dParMdPerp*factor) / (2.0*kappa)
        dw[1] = (dParMdPerp + 2.0*(dPar+dPerp)*kappa - dParMdPerp*factor) / (4.0*kappa)
        
        grad_dirs = self.protocol_hr['grad_dirs']
        G = self.protocol_hr['gradient_strength']
        delta = self.protocol_hr['delta']
        smalldel = self.protocol_hr['smalldel']
        fibredir = np.array([0, 0, 1])
        
        modQ = _GAMMA * smalldel * G
        bval = (delta - smalldel/3.0) * modQ**2
        
        cosTheta = np.dot(grad_dirs, fibredir)
        cosThetaSq = cosTheta**2
        
        E = np.exp(-bval * ((dw[0] - dw[1])*cosThetaSq + dw[1]))
        return E

## 7. Class: `NODDIIsotropic`

Core algorithm class.

Methods: `__init__`, `get_signal`

In [ ]:
class NODDIIsotropic:
    def __init__(self, scheme):
        self.scheme = scheme
        self.protocol_hr = _scheme2noddi(self.scheme)

    def get_signal(self, diff_iso):
        diff_iso *= 1e-6
        G = self.protocol_hr['gradient_strength']
        delta = self.protocol_hr['delta']
        smalldel = self.protocol_hr['smalldel']
        modQ = _GAMMA * smalldel * G
        bval = (delta - smalldel/3.0) * modQ**2
        return np.exp(-bval * diff_iso)

## 8. Forward Model

Define the forward operator mapping true model to observations.

```
y = M F x + n  (undersampled Fourier encoding)
```

Functions: `aux_structures_generate`, `create_high_resolution_scheme`, `generate_phantom_data`, `forward_operator`

In [ ]:
def aux_structures_generate(scheme, lmax=12):
    nSH = int((lmax + 1) * (lmax + 2) / 2)
    idx_IN = []
    idx_OUT = []
    for s in range(len(scheme.shells)):
        idx_IN.append(range(500 * s, 500 * (s + 1)))
        idx_OUT.append(range(nSH * s, nSH * (s + 1)))
    return idx_IN, idx_OUT

def create_high_resolution_scheme(scheme):
    n = len(scheme.shells)
    raw = np.zeros((500 * n, 7))
    row = 0
    for i in range(n):
        raw[row:row+500, 0:3] = GRAD_500
        if scheme.version == 0:
            raw[row:row+500, 3] = scheme.shells[i]['b']
        else:
            raw[row:row+500, 3] = scheme.shells[i]['G']
            raw[row:row+500, 4] = scheme.shells[i]['Delta']
            raw[row:row+500, 5] = scheme.shells[i]['delta']
            raw[row:row+500, 6] = scheme.shells[i]['TE']
        row += 500
        
    if scheme.version == 0:
        raw0 = np.zeros((500 * n, 4))
        row = 0
        for i in range(n):
            raw0[row:row+500, 0:3] = GRAD_500
            raw0[row:row+500, 3] = scheme.shells[i]['b']
            row += 500
        return Scheme(raw0)
    return Scheme(raw)

def generate_phantom_data():
    S = (20, 20, 1)
    bvals = np.hstack((np.zeros(1), np.ones(30) * 1000))
    bvecs = np.zeros((31, 3))
    bvecs[1:, :] = fibonacci_sphere(30)
    gtab = gradient_table(bvals, bvecs)
    
    np.savetxt('DWI.scheme', np.hstack((bvecs, bvals[:, None])), fmt='%.6f')
    
    from dipy.sims.voxel import multi_tensor
    data = np.zeros(S + (len(bvals),))
    GT_NDI = np.zeros(S)
    
    mevals = np.array([[0.0017, 0.0003, 0.0003], [0.0017, 0.0003, 0.0003]])
    
    for x in range(S[0]):
        for y in range(S[1]):
            if (x - S[0]/2)**2 + (y - S[1]/2)**2 < (S[0]/3)**2:
                angles = [(0, 0), (90, 0)]
                fractions = [50, 50]
                sig, _ = multi_tensor(gtab, mevals, S0=100, angles=angles, fractions=fractions, snr=None)
                sig = sig + np.random.randn(len(sig)) * (100/30)
                data[x, y, 0, :] = np.abs(sig)
                GT_NDI[x, y] = 0.7
            else:
                sig = np.ones(len(bvals)) * 100 * np.exp(-bvals * 0.003)
                sig = sig + np.random.randn(len(sig)) * (100/30)
                data[x, y, 0, :] = np.abs(sig)
                GT_NDI[x, y] = 0.0
    
    affine = np.eye(4)
    nib.save(nib.Nifti1Image(data, affine), 'DWI.nii')
    mask = np.ones(S, dtype=np.uint8)
    nib.save(nib.Nifti1Image(mask, affine), 'brain_mask.nii')
    nib.save(nib.Nifti1Image(GT_NDI, affine), 'GT_NDI.nii.gz')
    return 'DWI.nii', 'DWI.scheme', 'brain_mask.nii'

def forward_operator(scheme):
    # This generates the AMICO Kernels
    aux = precompute_rotation_matrices(lmax=12)
    idx_in, idx_out_gen = aux_structures_generate(scheme, lmax=12)
    idx_out_res, Ylm_out = aux_structures_resample(scheme, lmax=12)
    
    scheme_high = create_high_resolution_scheme(scheme)
    noddi_ic = NODDIIntraCellular(scheme_high)
    noddi_ec = NODDIExtraCellular(scheme_high)
    noddi_iso = NODDIIsotropic(scheme_high)
    
    dPar = 1.7E-3
    dIso = 3.0E-3
    IC_VFs = np.linspace(0.1, 0.99, 12)
    IC_ODs = np.hstack((np.array([0.03, 0.06]), np.linspace(0.09, 0.99, 10)))
    
    kernels_lut = []
    
    IC_KAPPAs = 1 / np.tan(IC_ODs * np.pi / 2)
    
    # Generate on sphere
    for kappa in IC_KAPPAs:
        signal_ic = noddi_ic.get_signal(dPar, kappa)
        for v_ic in IC_VFs:
            signal_ec = noddi_ec.get_signal(dPar, kappa, v_ic)
            signal = v_ic * signal_ic + (1 - v_ic) * signal_ec
            lm = rotate_kernel(signal, aux, idx_in, idx_out_gen, False, 500)
            kernels_lut.append(lm)
            
    signal_iso = noddi_iso.get_signal(dIso)
    lm_iso = rotate_kernel(signal_iso, aux, idx_in, idx_out_gen, True, 500)
    kernels_lut.append(lm_iso)
    
    # Resample to scheme
    nATOMS = len(kernels_lut)
    nS = scheme.nS
    
    KERNELS = {}
    KERNELS['wm'] = np.zeros((nATOMS - 1, 500, nS), dtype=np.float32)
    KERNELS['iso'] = np.zeros(nS, dtype=np.float32)
    KERNELS['icvf'] = np.zeros(nATOMS - 1)
    KERNELS['kappa'] = np.zeros(nATOMS - 1)
    KERNELS['norms'] = np.zeros((scheme.dwi_count, nATOMS - 1))
    
    idx = 0
    for i in range(len(IC_ODs)):
        for j in range(len(IC_VFs)):
            lm = kernels_lut[idx]
            k_resampled = resample_kernel(lm, nS, idx_out_res, Ylm_out, False, 500)
            KERNELS['wm'][idx, :, :] = k_resampled
            KERNELS['kappa'][idx] = 1.0 / np.tan(IC_ODs[i] * np.pi / 2.0)
            KERNELS['icvf'][idx] = IC_VFs[j]
            
            dwi_idx = scheme.dwi_idx
            atom_sig = KERNELS['wm'][idx, 0, dwi_idx]
            nrm = np.linalg.norm(atom_sig)
            KERNELS['norms'][:, idx] = 1.0 / nrm if nrm > 0 else 1.0
            idx += 1
            
    lm = kernels_lut[-1]
    KERNELS['iso'] = resample_kernel(lm, nS, idx_out_res, Ylm_out, True, 500)
    
    return KERNELS

## 9. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
min 1/2 ||MFx - y||^2 + lambda ||Psi x||_1  (compressed sensing)
```

Functions: `run_inversion`

In [ ]:
def run_inversion(data, mask, dirs, scheme, kernels):
    mask_indices = np.where(mask)
    y_data = data[mask_indices]
    d_dirs = dirs[mask_indices]
    n_voxels = y_data.shape[0]
    
    results = np.zeros((n_voxels, 3))
    
    from scipy.spatial import KDTree
    tree = KDTree(GRAD_500)
    
    lambda1 = 1e-3
    
    # Normalize data by b0 signal (standard in AMICO)
    b0_idx = scheme.b0_idx
    dwi_idx = scheme.dwi_idx
    
    for i in range(n_voxels):
        signal = y_data[i, :]
        direction = d_dirs[i, :]
        
        # Normalize by b0
        b0_signal = np.mean(signal[b0_idx])
        if b0_signal > 0:
            signal_norm = signal / b0_signal
        else:
            signal_norm = signal.copy()
        
        direction = direction / (np.linalg.norm(direction) + 1e-16)
        dist, idx_dir = tree.query(direction)
        dist2, idx_dir2 = tree.query(-direction)
        if dist2 < dist:
            idx_dir = idx_dir2
            
        A_wm = kernels['wm'][:, idx_dir, :].T
        A_iso = kernels['iso'][:, None]
        
        # Use ALL measurements including b0 for fitting
        # The b0 constraint (all atoms=1 at b0) helps disambiguate compartments
        
        # Normalize atoms for NNLS/LASSO stability
        norms_wm = np.linalg.norm(A_wm, axis=0)
        norms_wm[norms_wm == 0] = 1.0
        A_wm_scaled = A_wm / norms_wm[None, :]
        
        iso_norm = np.linalg.norm(A_iso[:, 0])
        iso_norm = iso_norm if iso_norm > 0 else 1.0
        A_iso_scaled = A_iso / iso_norm
        
        A_combined = np.hstack([A_wm_scaled, A_iso_scaled])
        
        # Use NNLS for more stable fitting than LASSO
        x_combined, _ = scipy.optimize.nnls(A_combined, signal_norm)
        
        x_wm_scaled = x_combined[:-1]
        x_iso_scaled = x_combined[-1]
        
        # Undo normalization
        x_wm = x_wm_scaled / norms_wm
        x_iso = x_iso_scaled / iso_norm
        
        sum_x_wm = np.sum(x_wm)
        total_sum = sum_x_wm + x_iso + 1e-16
        
        # NDI = volume fraction of intracellular compartment relative to TOTAL
        # Each WM atom represents (icvf * wm_signal + (1-icvf) * ec_signal)
        # The weighted icvf gives the IC fraction within WM
        # NDI = (weighted_icvf * wm_fraction)
        if sum_x_wm > 1e-16:
            weighted_icvf = np.sum(x_wm * kernels['icvf']) / sum_x_wm
            k1 = np.sum(x_wm * kernels['kappa']) / sum_x_wm
        else:
            weighted_icvf = 0.0
            k1 = 1.0
            
        wm_fraction = sum_x_wm / total_sum
        ndi = weighted_icvf * wm_fraction
        odi = (2.0 / np.pi) * np.arctan2(1.0, k1)
        fwf = x_iso / total_sum
        
        results[i, :] = [ndi, odi, fwf]
        
    # Reconstruct maps
    ndi_map = np.zeros(data.shape[:3])
    odi_map = np.zeros(data.shape[:3])
    fwf_map = np.zeros(data.shape[:3])
    
    ndi_map[mask_indices] = results[:, 0]
    odi_map[mask_indices] = results[:, 1]
    fwf_map[mask_indices] = results[:, 2]
    
    return ndi_map, odi_map, fwf_map

## 10. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(ndi_map, odi_map, fwf_map, mask):
    if os.path.exists('GT_NDI.nii.gz'):
        gt_ndi = nib.load('GT_NDI.nii.gz').get_fdata()
        rmse = np.sqrt(np.mean((ndi_map[mask] - gt_ndi[mask])**2))
        psnr = 20 * np.log10(1.0 / rmse) if rmse > 0 else float('inf')
        print(f"NDI RMSE: {rmse:.4f}")
        print(f"NDI PSNR: {psnr:.2f} dB")
    
    print(f"Mean NDI in mask: {np.mean(ndi_map[mask])}")
    print(f"Mean ODI in mask: {np.mean(odi_map[mask])}")
    
    # Center vs Background check
    S = ndi_map.shape
    center_mask = np.zeros(S, dtype=bool)
    for x in range(S[0]):
        for y in range(S[1]):
            if (x - S[0]/2)**2 + (y - S[1]/2)**2 < (S[0]/3)**2:
                center_mask[x, y, 0] = True
    
    mean_center_ndi = np.mean(ndi_map[center_mask])
    mean_bg_ndi = np.mean(ndi_map[mask & ~center_mask])
    
    print(f"Mean NDI in Center (Crossing): {mean_center_ndi:.3f}")
    print(f"Mean NDI in Background: {mean_bg_ndi:.3f}")

## 11. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
dwi_file = 'DWI.nii'
scheme_file = 'DWI.scheme'
mask_file = 'brain_mask.nii'

### 1. Load Data

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Load Data
data, mask, affine, scheme, dirs = load_and_preprocess_data(dwi_file, scheme_file, mask_file)

### 2. Forward Operator (Generate Kernels)

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. Forward Operator (Generate Kernels)
kernels = forward_operator(scheme)

### 3. Run Inversion

Intermediate processing step in the pipeline.

In [ ]:
# 3. Run Inversion
ndi_map, odi_map, fwf_map = run_inversion(data, mask, dirs, scheme, kernels)

### 4. Evaluate

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Evaluate
evaluate_results(ndi_map, odi_map, fwf_map, mask)

# Save results
nib.save(nib.Nifti1Image(ndi_map, affine), 'NODDI_NDI.nii.gz')
nib.save(nib.Nifti1Image(odi_map, affine), 'NODDI_ODI.nii.gz')
nib.save(nib.Nifti1Image(fwf_map, affine), 'NODDI_FWF.nii.gz')

# Save GT and Recon as .npy for evaluation
gt_ndi_img = nib.load('GT_NDI.nii.gz')
gt_ndi = gt_ndi_img.get_fdata()

# Ensure 3D shape matches
if gt_ndi.ndim == 2:
    gt_ndi = gt_ndi[:, :, np.newaxis]
if ndi_map.ndim == 2:
    ndi_map_3d = ndi_map[:, :, np.newaxis]
else:
    ndi_map_3d = ndi_map

np.save('gt_ndi.npy', gt_ndi)
np.save('recon_ndi.npy', ndi_map_3d)
np.save('gt_output.npy', gt_ndi)
np.save('recon_output.npy', ndi_map_3d)

# Compute PSNR/SSIM
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
gt_flat = gt_ndi.flatten()
recon_flat = ndi_map_3d.flatten()
data_range = max(gt_flat.max(), recon_flat.max()) - min(gt_flat.min(), recon_flat.min())
if data_range == 0:
    data_range = 1.0
psnr_val = peak_signal_noise_ratio(gt_ndi, ndi_map_3d, data_range=data_range)

gt_2d = gt_ndi[:, :, 0]
recon_2d = ndi_map_3d[:, :, 0]
ssim_val = structural_similarity(gt_2d, recon_2d, data_range=data_range)
print(f"\n=== FINAL METRICS ===")
print(f"PSNR: {psnr_val:.2f} dB")
print(f"SSIM: {ssim_val:.4f}")

# Save visualization
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(gt_2d, cmap='hot', vmin=0, vmax=0.7)
axes[0].set_title('GT NDI')
axes[0].axis('off')
axes[1].imshow(recon_2d, cmap='hot', vmin=0, vmax=0.7)
axes[1].set_title('Reconstructed NDI')
axes[1].axis('off')
diff = np.abs(gt_2d - recon_2d)
axes[2].imshow(diff, cmap='hot', vmin=0, vmax=0.3)
axes[2].set_title(f'|Difference| (PSNR={psnr_val:.2f}dB)')
axes[2].axis('off')
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/vis_result.png', dpi=150, bbox_inches='tight')
plt.close()
print("Visualization saved to results/vis_result.png")

print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 12. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **AMICO-master**:

1. **Problem Setup**: MRI acquires data in k-space. Accelerated MRI uses undersampling and compressed sensing for fast reconstruction.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=28.57 dB ← 🔧 修复前: 2.08 dB, SSIM=0.9066)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Accelerated Microstructure Imaging via Convex Optimization (AMICO) from diffusion MRI data
- Repository: https://github.com/daducci/AMICO
